## Create data for pytorch shadow model
(cf. qfm_pytorch_model_eval.ipynb)

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import cs_functions_jit as csj
import pennylane as qml
import pennylane.numpy as np
import numpy as onp
import jax
from jax import numpy as jnp


jax.config.update("jax_enable_x64", True)

path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 


import input_transform
importlib.reload(input_transform)
from input_transform import inverse_transform_clc

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)
import qnn_layouts_pennylane as pqcs



# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)
ind_features = [0,1,2,3,4,6]
### Architecture specifications
no_qubits = no_of_features


In [ ]:

### PQC architecture layout
# No of shots for circuit evaluation
no_shots = 1000 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'

### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    pqc_layout = pqcs.XYZ_circuit;  name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    pqc_layout = pqcs.ZZXY_circuit;  name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5; # best_exp = 6 # for inifinite shots
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

n_enc_name = str(n_enc)
n_dec_name = str(n_dec)

# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'
    filename_pars = ('optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + 
                     '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy')
    

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)


In [ ]:

### ---------------------------------------------------------------------------------------- ###
## ---------------------------------- Initialize QNN model ---------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

no_gate_angles = pqcs.no_of_angles_pqc(name_arch, no_qubits, n_enc, n_dec)
print(no_gate_angles)
no_params = no_gate_angles + no_qubits + 1

### Define the quantum device

if no_shots == 'inf':
    dev = qml.device("default.qubit.jax", wires=no_qubits)
else:   
    dev = qml.device("default.qubit.jax", wires=no_qubits, shots=no_shots)


### Define pqc with measured observables
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, pars):
    pqc_layout(inputs, pars, n_enc=n_enc, n_dec=n_dec, wires=dev.wires)
    return [qml.expval(qml.PauliZ(i)) for i in range(no_qubits)]

### Define the QNN model (pqc + postprocessing)
@jax.jit
def model_qnn(params, inputs):
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    print(no_angles)
    no_weights = no_qubits
    no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])
    print(weights)
    print(bias)

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]
    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)



In [ ]:


### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Start evaluations ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

no_angles = no_gate_angles
no_weights = no_qubits
no_bias = 1

weights = jax.lax.dynamic_slice(opt_params, [no_angles], [no_weights])
bias = jax.lax.dynamic_slice(opt_params, [no_angles + no_weights], [no_bias])


In [ ]:

@jax.jit
def linear_pre(x): #maps to [-pi,pi] or [0,pi], here identity
    y = x
    return y

@jax.jit
def linear_pre_inverse(y):
    x = y
    return x


In [ ]:
#Take model and separate it st. input and output dimensions match

@jax.jit
def model_qnn_q(params, inputs): #from Lorenzos Code
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    #no_weights = no_qubits
    #no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    #weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    #bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    #weighted_output = weights[0] * measured_batches[0]
    #for i in range(1,no_qubits):
    #    weighted_output = weighted_output + weights[i] * measured_batches[i]
    #predictions = weighted_output + bias
    return measured_batches#jnp.squeeze(predictions)


@jax.jit
def model_qnn_c(params,measured_batches): #from lorenzos code
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    no_weights = no_qubits
    no_bias = 1
    #angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    #measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]

    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)

In [ ]:
n0 = n_enc 
d = no_qubits 
Of = np.fft.fftfreq(2*n0+1,1/(2*n0+1))

freqs =[Of]*d

grids = np.meshgrid(*freqs,indexing = 'ij')
Omegafft = np.stack(grids,axis = -1)

shape = (2*n0+1,)*d+(d,)

Oflat =Omegafft.reshape(-1,d) #flatten
Xft = 2.*np.pi/(2*n0+1)*Oflat #compute support points





In [ ]:
def post_processing(out_batch_old,opt_params): #post processing is hard-coded into the pytorch model
        outs_batch_new = np.zeros((out_batch_old.shape[0],))
        batch_size = 100
        no_batches_test = int(np.floor(out_batch_old.shape[0]/ batch_size)) ###<----------!!!
        for kk in range(0, no_batches_test-1):
    
            outs_batch =  model_qnn_c(opt_params,out_batch_old[kk*batch_size:(kk+1)*batch_size,:].T)
    
            outs_batch = np.asarray(outs_batch)
            outs_batch = np.squeeze(outs_batch)
            # Clip values in [0,1]
            outs_batch = np.minimum(outs_batch, 1.0)
            outs_batch = np.maximum(outs_batch, 0.0)
            # If the output has been transformed, re-transform it back
            if transform_output: #here not yet included in dft
                 outs_batch = inverse_transform_clc(outs_batch)
            outs_batch_new[kk*batch_size:(kk+1)*batch_size] = np.copy(outs_batch)
            
        outs_batch =  model_qnn_c(opt_params,out_batch_old[(no_batches_test-1)*batch_size:,:].T)
    
        outs_batch = np.asarray(outs_batch)
        outs_batch = np.squeeze(outs_batch)
        # Clip values in [0,1]
        outs_batch = np.minimum(outs_batch, 1.0)
        outs_batch = np.maximum(outs_batch, 0.0)
            # If the output has been transformed, re-transform it back
        if transform_output: 
                 outs_batch = inverse_transform_clc(outs_batch)
        outs_batch_new[(no_batches_test-1)*batch_size:] = np.copy(outs_batch)

        return outs_batch_new


In [ ]:

trunc_frequencies = [50] + [i for i in onp.arange(500,5000,100)]+[i for i in onp.arange(5000,10000,2500)] + [10000,12500,15000]


# Compute Fourier Coefficients and truncate

In [ ]:
Cft_np = csj.compute_fourier_coeff_fft(model_qnn_q,Oflat,Xft,shape,opt_params, linear_pre_inverse)
print('done computing coefficients')
Cft_flat= Cft_np.reshape(-1,d)
Omegacutfft,Ccos, Csin = csj.find_largest_indices_real_trunc(shape,Omegafft, Cft_np,d,n0,trunc_frequencies) 
print('done truncating')




# Save parameters for pytorch model

In [ ]:
qfm = {'trunc_frequencies': onp.asarray(trunc_frequencies), 'spectrum': onp.asarray(Omegacutfft), 'Cos':  onp.asarray(Ccos), 'Sin':  onp.asarray(Csin)}
qfm['opt_params'] = onp.asarray(opt_params)
qfm['weights'] = onp.asarray(weights)
qfm['bias'] = onp.asarray(bias)
onp.save('qfm_' + str(no_qubits) +'f_'+ name_arch + '_'+ str(no_shots_training)+'t_'+ str(no_shots) +'s_test'+str(best_exp),qfm)
